# `klygo.mining` — start here

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/mining/Document_Mining_Overview.ipynb)

Analyze, deduplicate, cluster, balance, and select complete images with classical features.

In [1]:
# Install Klygo from the v2-release branch
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

## Choose a guide

| Submodule | Notebook | Colab | Use it for | APIs |
|---|---|---|---|---:|
| `analysis.py` | [Document_Mining_Analysis.ipynb](Document_Mining_Analysis.ipynb) | [Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/mining/Document_Mining_Analysis.ipynb) | Summarize statistics, compute multi-criteria scores, and compare datasets | 6 |
| `duplicates.py` | [Document_Mining_Duplicates.ipynb](Document_Mining_Duplicates.ipynb) | [Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/mining/Document_Mining_Duplicates.ipynb) | Detect exact and perceptual duplicate groups and deduplicate frames | 3 |
| `curation.py` | [Document_Mining_Curation.ipynb](Document_Mining_Curation.ipynb) | [Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/mining/Document_Mining_Curation.ipynb) | Cluster visual features, select diverse subsets, and balance class distributions | 4 |

## Try the public API

Load a YOLO dataset as `Detections`, then pass it directly to `mining.stats`, `mining.score`, `mining.deduplicate`, `mining.cluster`, `mining.select`, `mining.balance`, and `mining.compare`.

In [2]:
from klygo import archive, datasets, files, mining

workspace = files.join("artifacts", "tutorials", "mining")
flat_dir = files.join(workspace, "flat_coco8")
if not files.exists(files.join(flat_dir, "data.yaml")):
    files.mkdir(workspace)
    zip_path = files.download("https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip", workspace, overwrite=True, verbose=False)
    yaml_path = files.download("https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/datasets/coco8.yaml", workspace, overwrite=True, verbose=False)
    raw_dir = files.join(workspace, "raw_coco8")
    archive.extract(zip_path, raw_dir, overwrite=True, verbose=False)
    files.copy(yaml_path, files.join(raw_dir, "coco8.yaml"), overwrite=True)
    datasets.convert(
        source=raw_dir,
        target=flat_dir,
        tree={
            "images": ["coco8/images/train", "coco8/images/val"],
            "labels": ["coco8/labels/train", "coco8/labels/val"],
            "metadata": "coco8.yaml",
        },
        on_conflict="rename",
        overwrite=True,
    )

data = datasets.load(flat_dir, format="yolo", verbose=False)
print(f"Loaded dataset: {len(data)} images, {data.total_objects} boxes, {len(data.names)} classes")

report = mining.stats(data)
report.show()
scores = mining.score(data)
curated = mining.select(data, method="weighted", scores=scores, limit=4)
mining.compare(data, curated).show()

Loaded dataset: 8 images, 30 boxes, 12 classes
                                  MiningStats                                  
┌───────────────────┬─────────────────────────────────────────────────────────┐
│ Metric            │ Value                                                   │
├───────────────────┼─────────────────────────────────────────────────────────┤
│ images            │ 8                                                       │
│ boxes             │ 30                                                      │
│ classes           │ 12                                                      │
│ empty_images      │ 0                                                       │
│ multilabel_images │ 5                                                       │
│ images_per_class  │ {'bowl': 1, 'broccoli': 1, 'dog': 1, 'elephant': 1,     │
│                   │ 'giraffe': 1, 'horse': 1, 'orange': 1, 'person': 3,     │
│                   │ 'potted plant': 2, 'umbrella': 1, 'vase': 1, 'zebra

## Remember

Mining always selects complete images. Boxes and labels are features, not selection units. Inputs remain unchanged; use `Detections.export()` to write the selected dataset.